# Notebook 14 – EDA Mini Challenge




**Instructions:** update `DATA_PATH`; run each cell; replace the example interpretations with evidence from your dataset. Do not fully preprocess the data.

## 1. Load the dataset

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Option A: place your dataset CSV in the same folder and update this path.
DATA_PATH = "your_dataset.csv"

# Option B: if the file is not available, create a small example dataset so every cell can run.
if Path(DATA_PATH).exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded dataset: {DATA_PATH}")
else:
    rng = np.random.default_rng(42)
    n = 200
    df = pd.DataFrame({
        "customer_age": rng.integers(18, 70, n),
        "monthly_income": rng.normal(45000, 15000, n).round(0),
        "monthly_spend": rng.normal(18000, 7000, n).round(0),
        "visits_per_month": rng.poisson(4, n),
        "membership": rng.choice(["Basic", "Silver", "Gold"], n, p=[0.5, 0.3, 0.2]),
        "region": rng.choice(["North", "South", "East", "West"], n),
        "churn": rng.choice(["Yes", "No"], n, p=[0.25, 0.75])
    })
    # Add a few realistic data-quality issues for demonstration.
    df.loc[[3, 17, 41], "monthly_income"] = np.nan
    df.loc[[8, 55], "region"] = np.nan
    df.loc[12, "monthly_spend"] = 150000
    df = pd.concat([df, df.iloc[[0]]], ignore_index=True)
    print("Example dataset created because DATA_PATH was not found.")
print("Dataset shape:", df.shape)
display(df.head())

## 2. Understand the dataset
Record the dataset source, domain, likely use case, and target if known.

In [ ]:
DATASET_SOURCE = "Update this"
DOMAIN = "Investigate and update this"
TARGET_COLUMN = None  # Set a column name if a target is available.

print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head(10))
display(df.sample(min(5, len(df)), random_state=42))
print("Duplicate rows:", int(df.duplicated().sum()))

## 3. Inspect structure and data types

In [ ]:
df.info()
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "unique_values": df.nunique(dropna=True),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean()*100).round(2)
}))

## 4. Missing values and duplicates

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean()*100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing_summary[missing_summary["missing_count"] > 0])
print("Exact duplicate rows:", int(df.duplicated().sum()))
# Do not drop rows here. Investigate whether duplicates are legitimate.

## 5. Numerical feature analysis

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns:", numeric_cols)
if numeric_cols:
    display(df[numeric_cols].describe().T)
    for col in numeric_cols:
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
        sns.histplot(data=df, x=col, kde=True, ax=axes[0])
        axes[0].set_title(f"Distribution of {col}")
        sns.boxplot(data=df, x=col, ax=axes[1])
        axes[1].set_title(f"Box plot of {col}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric columns detected.")

## 6. Categorical feature analysis

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("Categorical-like columns:", cat_cols)
for col in cat_cols:
    print(f"\n{col}: {df[col].nunique(dropna=True)} unique non-null values")
    display(df[col].value_counts(dropna=False).head(20).to_frame("count"))
    if df[col].nunique(dropna=True) <= 30:
        plt.figure(figsize=(7, 3.5))
        df[col].value_counts(dropna=False).head(15).plot(kind="bar")
        plt.title(f"Category frequency: {col}")
        plt.ylabel("Count")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()

## 7. Outlier detection using IQR

In [ ]:
outlier_results = []
for col in numeric_cols:
    s = df[col].dropna()
    if s.empty:
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    mask = (s < lo) | (s > hi)
    outlier_results.append({"feature": col, "lower_bound": lo, "upper_bound": hi,
                            "potential_outliers": int(mask.sum()),
                            "percent": round(mask.mean()*100, 2)})
display(pd.DataFrame(outlier_results))
print("IQR flags potential outliers; validate them before deciding what to do.")

## 8. Distribution analysis

In [ ]:
if numeric_cols:
    distribution_summary = pd.DataFrame({
        "mean": df[numeric_cols].mean(),
        "median": df[numeric_cols].median(),
        "std": df[numeric_cols].std(),
        "skewness": df[numeric_cols].skew(),
        "kurtosis": df[numeric_cols].kurt()
    })
    display(distribution_summary.round(3))
else:
    print("No numeric features to analyze.")

## 9. Correlations and feature relationships

In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()
    plt.figure(figsize=(max(7, len(numeric_cols)), max(5, len(numeric_cols)*0.7)))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Correlation heatmap")
    plt.tight_layout()
    plt.show()
    display(corr.round(2))
else:
    print("Need at least two numeric columns for correlation analysis.")

# Optional: inspect numeric features by target if the target exists.
if TARGET_COLUMN and TARGET_COLUMN in df.columns:
    print("Target counts:")
    display(df[TARGET_COLUMN].value_counts(dropna=False).to_frame("count"))
    for col in numeric_cols:
        if col != TARGET_COLUMN:
            plt.figure(figsize=(7, 3.5))
            sns.boxplot(data=df, x=TARGET_COLUMN, y=col)
            plt.title(f"{col} by {TARGET_COLUMN}")
            plt.xticks(rotation=30)
            plt.tight_layout()
            plt.show()

## 10. Meaningful visualizations
Choose plots that answer a question. Avoid making charts without explaining their purpose.

In [ ]:
# Example relationship plot: select two numeric columns when available.
if len(numeric_cols) >= 2:
    x_col, y_col = numeric_cols[0], numeric_cols[1]
    plt.figure(figsize=(7, 4))
    sns.scatterplot(data=df, x=x_col, y=y_col)
    plt.title(f"{y_col} vs {x_col}")
    plt.tight_layout()
    plt.show()
    print(f"Question: Is there a visible relationship between {x_col} and {y_col}?")
else:
    print("Choose suitable columns and create a plot that answers a specific question.")